##Arabic Startup Idea Novelty Detection Using Semantic Similarity

**Ruba Aljuhani**

**EMAIL :**
Ruba35@gmail.com

**AND YOU CAN SEE THE FULL PROJECT ON :**
[GITHUB](https://github.com/ii3ruj/Arabic-Idea-Novelty-Detection-Using-Semantic-Similarity/tree/main)

In [ ]:
!pip install pandas openpyxl scikit-learn

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd

df = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/3750_hackathon_ideas.csv")

df.head()

In [ ]:
df = df.dropna()
print("After removing nulls:", len(df))

In [ ]:
df = df.drop_duplicates()
print("After removing duplicates:", len(df))

## Feature Selection

In this step, i select the project description column because it contains the most meaningful textual information for semantic similarity analysis.

In [ ]:
ideas = df["الوصف"].tolist()

print(ideas[:3])

## Model 1: Bag of Words + Cosine Similarity

In this model, i represent each idea using word frequency counts.
Then, cosine similarity is used to compare a new idea with existing ideas in the dataset.


In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
vectorizer = CountVectorizer()

X = vectorizer.fit_transform(ideas)

print(X.shape)

In [ ]:
new_idea = ["ضع فكرتك هنا "]

new_vec = vectorizer.transform(new_idea)

In [ ]:
similarities = cosine_similarity(new_vec, X)

max_similarity = similarities.max()

novelty = 1 - max_similarity

In [ ]:
def get_novelty_level(novelty):
    if novelty < 0.30:
        return "Low Novelty"
    elif novelty < 0.60:
        return "Medium Novelty"
    else:
        return "High Novelty"

level = get_novelty_level(novelty)

print("Model: Bag of Words")
print("-------------------")
print("Result:")
print("Similarity:", round(max_similarity, 2))
print("Novelty:", round(novelty, 2))
print("Level:", level)

## Model 2: TF-IDF + Cosine Similarity
TF-IDF improves text representation by giving higher importance to informative words and reducing the impact of very common words.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [ ]:
tfidf_vectorizer = TfidfVectorizer()

X_tfidf = tfidf_vectorizer.fit_transform(ideas)

print(X_tfidf.shape)

In [ ]:
new_idea = ["ضع فكرتك هنا "]

new_vec_tfidf = tfidf_vectorizer.transform(new_idea)

In [ ]:
similarities_tfidf = cosine_similarity(new_vec_tfidf, X_tfidf)

max_similarity_tfidf = similarities_tfidf.max()

novelty_tfidf = 1 - max_similarity_tfidf

level_tfidf = get_novelty_level(novelty_tfidf)

print("Model: TF-IDF")
print("-------------------")
print("Result:")
print("Similarity:", round(max_similarity_tfidf, 2))
print("Novelty:", round(novelty_tfidf, 2))
print("Level:", level_tfidf)

## Model 3: Sentence-BERT + Cosine Similarity

Sentence-BERT is a modern NLP model that converts sentences into semantic embeddings.
Unlike Bag of Words and TF-IDF, it can capture the meaning of the idea, not only the exact words.

In [ ]:
!pip install sentence-transformers

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
sbert_model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

In [ ]:
sbert_embeddings = sbert_model.encode(ideas, show_progress_bar=True)

In [ ]:
new_idea = ["ضع فكرتك هنا "]

new_embedding = sbert_model.encode(new_idea)

similarities_sbert = cosine_similarity(new_embedding, sbert_embeddings)

max_similarity_sbert = similarities_sbert.max()
novelty_sbert = 1 - max_similarity_sbert
level_sbert = get_novelty_level(novelty_sbert)

print("Model: SBERT")
print("-------------------")
print("Result:")
print("Similarity:", round(max_similarity_sbert, 2))
print("Novelty:", round(novelty_sbert, 2))
print("Level:", level_sbert)

## Model 4: AraBERT + Semantic Similarity

AraBERT is a transformer model specifically trained on Arabic text.
It is designed to better understand Arabic language structure and semantic meaning compared to multilingual models.

In [ ]:
!pip install transformers torch

In [ ]:
from transformers import AutoTokenizer, AutoModel
import torch
import numpy as np

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("aubmindlab/bert-base-arabertv02")

arabert_model = AutoModel.from_pretrained("aubmindlab/bert-base-arabertv02")

In [ ]:
def get_embedding(text):

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )

    with torch.no_grad():
        outputs = arabert_model(**inputs)

    embedding = outputs.last_hidden_state.mean(dim=1)

    return embedding.numpy()

In [ ]:
arabert_embeddings = []

for idea in ideas:

    emb = get_embedding(idea)

    arabert_embeddings.append(emb[0])

In [ ]:
new_idea = "ضع فكرتك هنا "
new_embedding = get_embedding(new_idea)

In [ ]:
similarities_arabert = cosine_similarity(
    new_embedding,
    arabert_embeddings
)

max_similarity_arabert = similarities_arabert.max()

novelty_arabert = 1 - max_similarity_arabert

level_arabert = get_novelty_level(novelty_arabert)

print("Model: AraBERT")
print("-------------------")
print("Result:")
print("Similarity:", round(max_similarity_arabert, 2))
print("Novelty:", round(novelty_arabert, 2))
print("Level:", level_arabert)

In [ ]:
results = pd.DataFrame({
    "Model": ["Bag of Words", "TF-IDF", "SBERT", "AraBERT"],
    "Similarity": [
        max_similarity,
        max_similarity_tfidf,
        max_similarity_sbert,
        max_similarity_arabert
    ],
    "Novelty": [
        novelty,
        novelty_tfidf,
        novelty_sbert,
        novelty_arabert
    ],
    "Level": [
        level,
        level_tfidf,
        level_sbert,
        level_arabert
    ]
})

results

In [ ]:
results.to_csv("/content/drive/MyDrive/model_comparison_results.csv", index=False)